In [ ]:
import sys
sys.path.insert(0, "..")

import glob
import yaml
import numpy as np
import requests
from datetime import date
from pathlib import Path
from tqdm import tqdm

from src.data.combine_goes import combine_all
from src.data.download_imerg import download_imerg_date_range
from src.data.download_modis import download_modis_date_range
from src.data.download_goes import download_goes_cloud_products


# Load configs
with open("../configs/config-dual-branch-unet.yaml") as f:
    config = yaml.safe_load(f)

with open("../credentials.yaml") as f:
    creds = yaml.safe_load(f)

IMERG_DIR       = config["data"]["imerg_dir"]
MODIS_DIR       = config["data"]["modis_dir"]
GOES_DIR        = config["data"]["goes_dir"]
STATION_CSV_DIR = config["data"]["station_csv_dir"]
DEM_PATH        = config["data"]["dem_path"]
CACHE_DIR       = config["data"]["cache_dir"]
BBOX            = config["data"]["hawaii_bbox"]

TRAIN_YEARS  = config["training"]["train_years"]
VAL_MONTHS   = config["training"]["val_months"]    # {year: 2021, start_month: 1, end_month: 3}
TEST_MONTHS  = config["training"]["test_months"]   # {year: 2021, start_month: 4, end_month: 6}
VAL_YEAR     = VAL_MONTHS["year"]
TEST_YEAR    = TEST_MONTHS["year"]

PPS_EMAIL    = creds["pps"]["email"]
PPS_PASSWORD = creds["pps"]["password"]
ED_USER      = creds["earthdata"]["username"]
ED_PASS      = creds["earthdata"]["password"]

ALL_YEARS  = list(set(TRAIN_YEARS + [VAL_YEAR, TEST_YEAR]))
START_DATE = date(min(ALL_YEARS), 1, 1)
END_DATE   = date(max(ALL_YEARS), 12, 31)

In [2]:
## Download HCDP Station Rainfall Data
HCDP_STATION_BASE = (
    "https://ikeauth.its.hawaii.edu/files/v2/download/public/system/"
    "ikewai-annotated-data/HCDP/production/rainfall/new/day/statewide/"
    "partial/station_data"
)

for year in ALL_YEARS:
    for month in range(1, 13):
        fname = f"rainfall_new_day_statewide_partial_station_data_{year:04d}_{month:02d}.csv"
        out_path = Path(STATION_CSV_DIR) / f"{year:04d}" / f"{month:02d}" / fname
        
        if out_path.exists():
            continue
        
        out_path.parent.mkdir(parents=True, exist_ok=True)
        url = f"{HCDP_STATION_BASE}/{year:04d}/{month:02d}/{fname}"
        
        r = requests.get(url, timeout=30)
        if r.status_code == 200:
            out_path.write_bytes(r.content)
        else:
            print(f"Missing: {year}-{month}")

In [6]:
## Download IMERG Data  
download_imerg_date_range(
    start=START_DATE,
    end=END_DATE,
    out_dir=Path(IMERG_DIR),
    pps_email=PPS_EMAIL,
    pps_password=PPS_PASSWORD,
)

                                                                     
IMERG download:  64%|██████▍   | 470/731 [1:59:55<4:02:07, 55.66s/it]

                                                                     
IMERG download:  64%|██████▍   | 470/731 [2:40:38<4:02:07, 55.66s/it]

IMERG download: 100%|██████████| 731/731 [1:17:48<00:00,  6.39s/it]


[PosixPath('../data/IMERG/3B-HHR-E.MS.MRG.3IMERG.20200101-S233000-E235959.1410.V07B.1day.tif'),
 PosixPath('../data/IMERG/3B-HHR-E.MS.MRG.3IMERG.20200102-S233000-E235959.1410.V07B.1day.tif'),
 PosixPath('../data/IMERG/3B-HHR-E.MS.MRG.3IMERG.20200103-S233000-E235959.1410.V07B.1day.tif'),
 PosixPath('../data/IMERG/3B-HHR-E.MS.MRG.3IMERG.20200104-S233000-E235959.1410.V07B.1day.tif'),
 PosixPath('../data/IMERG/3B-HHR-E.MS.MRG.3IMERG.20200105-S233000-E235959.1410.V07B.1day.tif'),
 PosixPath('../data/IMERG/3B-HHR-E.MS.MRG.3IMERG.20200106-S233000-E235959.1410.V07B.1day.tif'),
 PosixPath('../data/IMERG/3B-HHR-E.MS.MRG.3IMERG.20200107-S233000-E235959.1410.V07B.1day.tif'),
 PosixPath('../data/IMERG/3B-HHR-E.MS.MRG.3IMERG.20200108-S233000-E235959.1410.V07B.1day.tif'),
 PosixPath('../data/IMERG/3B-HHR-E.MS.MRG.3IMERG.20200109-S233000-E235959.1410.V07B.1day.tif'),
 PosixPath('../data/IMERG/3B-HHR-E.MS.MRG.3IMERG.20200110-S233000-E235959.1410.V07B.1day.tif'),
 PosixPath('../data/IMERG/3B-HHR-E.MS.MR

In [3]:
## Download GOES ABI Cloud Products (2018-2022)
# Satellite mapping:
#   GOES-17  → 2018-2022  (Western US / Hawaii operational satellite)
#   GOES-18  → 2023+      (replaced GOES-17 on 2023-01-04)
#
# Four synoptic times per day (UTC): 00:00, 06:00, 12:00, 18:00
# Product: ABI-L2-ACMF = Advanced Clear Sky Mask, Full Disk
#   Other cloud products you can pass as `product=`:
#     "ABI-L2-ACHTF"  – Cloud Top Height, Full Disk
#     "ABI-L2-CTPF"   – Cloud Top Phase, Full Disk
#     "ABI-L2-TPWF"   – Total Precipitable Water, Full Disk
#
# No AWS credentials needed — NOAA data are publicly accessible.

import sys
sys.path.insert(0, "..")

from datetime import date
from pathlib import Path
from src.data.download_goes import download_goes_cloud_products

GOES_START = date(2021, 1, 1)
GOES_END   = date(2021, 12, 30)

download_goes_cloud_products(
    start=GOES_START,
    end=GOES_END,
    out_dir=Path("../data/GOES"),
    product="ABI-L2-ACMF",
    target_hours=(0, 6, 12, 18),
)

GOES ABI-L2-ACMF:  15%|█▌        | 55/364 [42:24<3:35:09, 41.78s/day]

GOES ABI-L2-ACMF:  23%|██▎       | 82/364 [1:03:10<3:38:05, 46.40s/day]

GOES ABI-L2-ACMF:  49%|████▉     | 179/364 [2:17:00<2:39:04, 51.59s/day]

GOES ABI-L2-ACMF:  49%|████▉     | 180/364 [2:17:01<2:08:29, 41.90s/day]

GOES ABI-L2-ACMF:  55%|█████▌    | 202/364 [2:36:34<2:27:41, 54.70s/day]

...

GOES ABI-L2-ACMF:  56%|█████▌    | 203/364 [2:36:35<1:52:37, 41.97s/day]

GOES ABI-L2-ACMF:  56%|█████▌    | 203/364 [2:36:35<1:52:37, 41.97s/day]

GOES ABI-L2-ACMF:  56%|█████▌    | 203/364 [2:36:35<1:52:37, 41.97s/day]

GOES ABI-L2-ACMF: 100%|██████████| 364/364 [6:35:40<00:00, 65.22s/day]   


[PosixPath('../data/GOES/goes17/2021/001/00/OR_ABI-L2-ACMF-M6_G17_s20210010000320_e20210010009386_c20210010010004.nc'),
 PosixPath('../data/GOES/goes17/2021/001/06/OR_ABI-L2-ACMF-M6_G17_s20210010600320_e20210010609387_c20210010610013.nc'),
 PosixPath('../data/GOES/goes17/2021/001/12/OR_ABI-L2-ACMF-M6_G17_s20210011200321_e20210011209387_c20210011210012.nc'),
 PosixPath('../data/GOES/goes17/2021/001/18/OR_ABI-L2-ACMF-M6_G17_s20210011800321_e20210011809388_c20210011810013.nc'),
 PosixPath('../data/GOES/goes17/2021/002/00/OR_ABI-L2-ACMF-M6_G17_s20210020000319_e20210020009386_c20210020010012.nc'),
 PosixPath('../data/GOES/goes17/2021/002/06/OR_ABI-L2-ACMF-M6_G17_s20210020600320_e20210020609387_c20210020610011.nc'),
 PosixPath('../data/GOES/goes17/2021/002/12/OR_ABI-L2-ACMF-M6_G17_s20210021200321_e20210021209387_c20210021209594.nc'),
 PosixPath('../data/GOES/goes17/2021/002/18/OR_ABI-L2-ACMF-M6_G17_s20210021800321_e20210021809388_c20210021810018.nc'),
 PosixPath('../data/GOES/goes17/2021/003

In [4]:
## Combine GOES daily snapshots (max aggregation over 00, 06, 12, 18 UTC)
# Reads the 4 raw hourly .nc files per day downloaded above and produces
# one combined file per day under:
#   <GOES_DIR>/combined/<satellite>/<year>/goes_<year>_<doy>.nc
#
# Each combined file contains:
#   BCM  – shape (5424, 5424)  max Binary Cloud Mask  (0=clear, 1=cloudy)
#   DQF  – shape (5424, 5424)  max Data Quality Flag
#   y, x – geostationary grid coordinates
# Re-running is safe — existing files are skipped unless overwrite=True.

# from src.data.combine_goes import combine_all

combine_all(
    goes_dir=Path(GOES_DIR),
    out_dir=Path(GOES_DIR) / "combined",
    overwrite=False,
)

Combining GOES days:   0%|          | 0/730 [00:00<?, ?day/s]

Combining GOES days: 100%|██████████| 730/730 [04:54<00:00,  2.48day/s] 


Done. Written: 364  Skipped (already exist): 366
Output directory: ../data/GOES/combined
